In [ ]:
import torch
from torch import optim
from torch import nn
from torch.utils.data import Dataset, DataLoader, random_split
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

!pip install -q kaggle
files.upload()
!kaggle datasets download harshadapatil31/student-performance-and-study-habits-dataset
!unzip -q student-performance-and-study-habits-dataset.zip

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/harshadapatil31/student-performance-and-study-habits-dataset
License(s): CC0-1.0
100% 14.7k/14.7k [00:00<00:00, 26.7MB/s]



In [ ]:
df = pd.read_csv("/content/student_performance_dataset.csv")
df.head()

df = df[["study_time_hours", "attendance_percent", "final_exam_score"]]
a = df.iloc[:, 2].values # 넘파이
df.head()

print(type(a))
a= torch.tensor(a)
print(type(a))

<class 'numpy.ndarray'>
<class 'torch.Tensor'>


In [ ]:
class DS(Dataset):
  def __init__(self, df):
    self.df = df
    self.x = self.df.iloc[:, 0:2].values
    self.y = self.df.iloc[:, 2].values
    self.length = len(self.df)

  def __getitem__(self, index):
    x = torch.FloatTensor([self.x[index]])
    y = torch.FloatTensor([self.y[index]])
    return x, y

  def __len__(self): # 필수
    return self.length

class LM(nn.Module):
  def __init__(self):
    super().__init__()
    self.layer = nn.Linear(2,1)

  def forward(self, x):
    x = self.layer(x)
    return x

class Trainer():
  def __init__(self, ds):
    self.ds = ds
    self.dataset = DS(self.ds)
    self.dataloader = DataLoader(self.dataset, batch_size = 10, shuffle = True, drop_last = True)
    self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
    self.model = LM()
    self.criterion = nn.MSELoss().to(self.device)
    self.lr = 0.01
    self.optimizer = optim.Adam(self.model.parameters(), lr = self.lr)
    self.epoch = 200

  def train(self):
    for step in range(self.epoch):

      for x, y in self.dataloader:
        x = x.to(self.device)
        y = y.to(self.device)

        output = self.model(x)
        loss = self.criterion(output, y)

        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        cost += loss.item()

      if (step+1) % 10 == 0:
        print(f"epoch: {step+1}, loss: {(cost.item() / len(self.dataset)):.2f}")

    torch.save(self.model, 'model.pt')


MyTrainer = Trainer(df)
MyTrainer.train()




epoch: 10, loss: 17.62
epoch: 20, loss: 17.10
epoch: 30, loss: 16.87
epoch: 40, loss: 16.29
epoch: 50, loss: 16.37
epoch: 60, loss: 15.93
epoch: 70, loss: 15.54
epoch: 80, loss: 15.13
epoch: 90, loss: 14.96
epoch: 100, loss: 14.92
epoch: 110, loss: 14.42
epoch: 120, loss: 14.27
epoch: 130, loss: 14.00
epoch: 140, loss: 13.80
epoch: 150, loss: 13.66
epoch: 160, loss: 13.63
epoch: 170, loss: 13.37
epoch: 180, loss: 13.07
epoch: 190, loss: 13.17
epoch: 200, loss: 12.86


In [ ]:
model = torch.load("model.pt", map_location = 'cuda' if torch.cuda.is_available() else 'cpu', weights_only = False)
print(round(model(torch.FloatTensor([[6, 100]])).item(),2))

92.3
